In [6]:
import duckdb
import matplotlib.pyplot as plt

con = duckdb.connect("../data/m5.db")

with open("../queries/create_base_dataset_weekly.sql") as f:
    query = f.read()

df = con.execute(query).df()

print(df.shape)
print(df.info())
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(8476220, 14)
<class 'pandas.DataFrame'>
RangeIndex: 8476220 entries, 0 to 8476219
Data columns (total 14 columns):
 #   Column           Dtype         
---  ------           -----         
 0   id               str           
 1   item_id          str           
 2   dept_id          str           
 3   cat_id           str           
 4   store_id         str           
 5   state_id         str           
 6   wm_yr_wk         int64         
 7   week_start_date  datetime64[us]
 8   week_end_date    datetime64[us]
 9   n_events_1       int64         
 10  n_events_2       int64         
 11  snap_days        float64       
 12  sales            float64       
 13  avg_sell_price   float64       
dtypes: datetime64[us](2), float64(3), int64(3), str(6)
memory usage: 905.4 MB
None


,id,item_id,dept_id,cat_id,store_id,state_id,wm_yr_wk,week_start_date,week_end_date,n_events_1,n_events_2,snap_days,sales,avg_sell_price
0,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,11101,2011-01-29,2011-02-04,0,0,4.0,10.0,2.0
1,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,11102,2011-02-05,2011-02-11,1,0,6.0,6.0,2.0
2,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,11103,2011-02-12,2011-02-18,1,0,0.0,10.0,2.0
3,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,11104,2011-02-19,2011-02-25,1,0,0.0,13.0,2.0
4,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,11105,2011-02-26,2011-03-04,0,0,4.0,15.0,2.0


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8476220 entries, 0 to 8476219
Data columns (total 14 columns):
 #   Column           Dtype         
---  ------           -----         
 0   id               str           
 1   item_id          str           
 2   dept_id          str           
 3   cat_id           str           
 4   store_id         str           
 5   state_id         str           
 6   wm_yr_wk         int64         
 7   week_start_date  datetime64[us]
 8   week_end_date    datetime64[us]
 9   n_events_1       int64         
 10  n_events_2       int64         
 11  snap_days        float64       
 12  sales            float64       
 13  avg_sell_price   float64       
dtypes: datetime64[us](2), float64(3), int64(3), str(6)
memory usage: 905.4 MB


In [8]:
for col in df.select_dtypes(include=["str", "datetime"]).columns:
    print(df[col].value_counts())
    print("="*30)

id
FOODS_1_001_CA_1_evaluation        278
FOODS_1_001_CA_2_evaluation        278
FOODS_1_001_CA_3_evaluation        278
FOODS_1_001_CA_4_evaluation        278
FOODS_1_001_TX_1_evaluation        278
                                  ... 
HOUSEHOLD_2_516_TX_2_evaluation    278
HOUSEHOLD_2_516_TX_3_evaluation    278
HOUSEHOLD_2_516_WI_1_evaluation    278
HOUSEHOLD_2_516_WI_2_evaluation    278
HOUSEHOLD_2_516_WI_3_evaluation    278
Name: count, Length: 30490, dtype: int64
item_id
FOODS_1_001        2780
FOODS_1_002        2780
FOODS_1_003        2780
FOODS_1_004        2780
FOODS_1_005        2780
                   ... 
HOUSEHOLD_2_512    2780
HOUSEHOLD_2_513    2780
HOUSEHOLD_2_514    2780
HOUSEHOLD_2_515    2780
HOUSEHOLD_2_516    2780
Name: count, Length: 3049, dtype: int64
dept_id
FOODS_3        2287940
HOUSEHOLD_1    1478960
HOUSEHOLD_2    1431700
HOBBIES_1      1156480
FOODS_2        1106440
FOODS_1         600480
HOBBIES_2       414220
Name: count, dtype: int64
cat_id
FOODS        

In [9]:
for col in df.select_dtypes(include=["int", "float"]).columns:
    print(df[col].describe())
    print("="*30)

count    8.476220e+06
mean     1.134387e+04
std      1.532912e+02
min      1.110100e+04
25%      1.121800e+04
50%      1.133550e+04
75%      1.145200e+04
max      1.161700e+04
Name: wm_yr_wk, dtype: float64
count    8.476220e+06
mean     5.683453e-01
std      6.737671e-01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      1.000000e+00
max      3.000000e+00
Name: n_events_1, dtype: float64
count    8.476220e+06
mean     1.438849e-02
std      1.190859e-01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.000000e+00
Name: n_events_2, dtype: float64
count    8.476220e+06
mean     2.302158e+00
std      2.281312e+00
min      0.000000e+00
25%      0.000000e+00
50%      2.000000e+00
75%      4.000000e+00
max      7.000000e+00
Name: snap_days, dtype: float64
count    8.476220e+06
mean     7.895875e+00
std      2.362555e+01
min      0.000000e+00
25%      0.000000e+00
50%      2.000000e+00
75%      7.000000e+00
max      3.53

In [10]:
def plot_agg_id_sales(df, agg_id=None):
    if agg_id is None:
        agg_id = df["agg_id"].sample(1).iloc[0]
    
    df_sample = df[df["agg_id"] == agg_id].copy()
    
    plt.figure(figsize=(15, 6))
    plt.plot(df_sample["date"], df_sample["sales"])
    plt.title(f"Sales over time for agg_id: {agg_id}")
    plt.xlabel("Date")
    plt.ylabel("Sales")
    plt.xticks(rotation=45)
    plt.grid()
    plt.tight_layout()
    plt.show()

In [11]:
plot_agg_id_sales(df)

KeyError: 'agg_id'

In [ ]:
df_statistics = df.groupby("agg_id")["sales"].agg(["mean", "std", "min", "max"]).reset_index()

df_statistics['cv'] = df_statistics['std'] / df_statistics['mean']

print(df_statistics.nsmallest(10, 'cv').to_string(index=False))
print(df_statistics.nlargest(10, 'cv').to_string(index=False))

In [ ]:
plot_agg_id_sales(df, agg_id="HOUSEHOLD_2_HOUSEHOLD_CA_3_CA")
plot_agg_id_sales(df, agg_id="FOODS_2_FOODS_CA_2_CA")